In [6]:
!pip uninstall -y sagemaker

Found existing installation: sagemaker 3.21.0
Uninstalling sagemaker-3.21.0:
  Successfully uninstalled sagemaker-3.21.0


In [7]:
!pip install "sagemaker<3.0.0" --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 157.9 MB/s  0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [multiprocess]
      Successfully uninstalled multiprocess-0.70.16━━━━━━━━━━━ 1/4 [multiprocess]
  Attempting uninstall: sagemaker-core━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [multiprocess]
    Found existing installation: sagemaker-core 2.21.0━━━━━━━━ 1/4 [multiprocess]
    Uninstalling sagemaker-core-2.21.0:━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [multiprocess]
      Successfully uninstalled sagemaker-core-2.21.0━━━━━━━━━━ 1/4 [multiprocess]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sagemaker]/4 [sagemaker]core]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This

In [8]:
!which pip
!which python
!pip show sagemaker

~/anaconda3/envs/python3/bin/pip
~/anaconda3/envs/python3/bin/python
Name: sagemaker
Version: 2.257.6
Summary: Open source library for training and deploying models on Amazon SageMaker.
Home-page: https://github.com/aws/sagemaker-python-sdk
Author: Amazon Web Services
Author-email: 
License: 
Location: /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages
Requires: attrs, boto3, cloudpickle, docker, fastapi, google-pasta, graphene, importlib-metadata, jsonschema, numpy, omegaconf, packaging, pandas, pathos, platformdirs, protobuf, psutil, pytz, pyyaml, requests, sagemaker-core, schema, smdebug-rulesconfig, tblib, tqdm, urllib3, uvicorn
Required-by: 


In [1]:
import sagemaker


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [2]:
import sagemaker
print(sagemaker.__file__)
print(sagemaker.__version__)
from sagemaker.huggingface import HuggingFace

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py
2.257.6


In [3]:
role = sagemaker.get_execution_role()

In [4]:
role

'arn:aws:iam::371892604080:role/SageMakerLLMRole'

In [5]:
hyperparameters = {
    "model_id": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    "epochs": 2,
    "per_device_train_batch_size": 2,
    "lr": 2e-5
}

In [8]:
estimator = HuggingFace(
    entry_point="train.py",
    source_dir="./scripts",
    role=role,
    transformers_version="4.36",
    pytorch_version="2.1",
    py_version="py310",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    output_path="s3://llm-model-artifacts-ishu/models/",
    hyperparameters=hyperparameters
)

## Run only for the training

In [ ]:
estimator.fit({
    "train": "s3://llm-finetune-dataset-ishu/datasets/"
})

In [ ]:
 estimator.latest_training_job.model_data
 estimator.model_data

In [ ]:
#To check all the accessible services inside your AWS Sagemaker

# from sagemaker import image_uris

# image_uris.retrieve(
#     framework="huggingface",
#     region="Europe(Stockholm)",   
#     version="4.37.0",
#     image_scope="inference"
# )

In [ ]:
 # instance_type="ml.g5.xlarge",

In [ ]:
# model = HuggingFaceModel(
#     model_data="s3://bucket/model.tar.gz",
#     role=role,
#     entry_point="inference.py",
#     source_dir="inference",
#     transformers_version="4.36",
#     pytorch_version="2.1",
#     py_version="py310"
# )

In [ ]:
## the code for the deployment
import sagemaker
from sagemaker.huggingface import HuggingFaceModel

role = sagemaker.get_execution_role()

model = HuggingFaceModel(
    model_data=  # example path "s3://llm-model-artifacts-ishu/models/huggingface-pytorch-training-2026-09-13-17-05-14-536/output/model.tar.gz",
    role=role,
    transformers_version="4.37.0",
    pytorch_version="2.1.0",
    py_version="py310",
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name="live-finetune-endpoint"
)


In [ ]:
## THIS IS JUST TO VALIDATE WHETHER MODEL WORKING OR NOT IN THE NOTEBOOK ITSELF
predictor.predict({"inputs": "Explain AWS S3"})

In [ ]:
## after the deployment URL will look like this
https://runtime.sagemaker.<region>.amazonaws.com/endpoints/live-finetune-endpoint/invocations

In [ ]:
import boto3, json

runtime = boto3.client("sagemaker-runtime", region_name="ap-south-1")

resp = runtime.invoke_endpoint(
    EndpointName="live-finetune-my-endpoint",
    ContentType="application/json",
    Body=json.dumps({"inputs": "hello"})
)

print(resp["Body"].read().decode())
